# ASR RTF Benchmark

Đo **RTF (Real-Time Factor)** cho model ASR hiện tại trên A100.

- **Model**: ZipFormer-RNNT (sherpa-onnx, OfflineRecognizer)
- **Dataset**: 6000 WAV từ `doof-ferb/vlsp2020_vinai_100h`
- **Metrics**: RTF P50, RTF P95, Latency P50, P95
- **Output**: `Drive/asr_benchmark/asr_rtf_results/`

## Trước khi chạy

Upload lên Google Drive:
```
Drive/asr_benchmark/samples/                    ← 6000 WAV (evaluation/asr_benchmark_samples/)
Drive/asr_benchmark/asr_benchmark_samples.jsonl ← metadata
Drive/asr_benchmark/model_data/                 ← nutrition-callbot/asr/data/ (4 file: encoder/decoder/joiner .onnx + config.json)
```
Chọn Runtime: **A100 GPU**

## RTF
```
RTF = processing_time / audio_duration
```
RTF < 1.0 → real-time capable. RTF P95 phản ánh worst-case.

In [ ]:
# Install dependencies
!pip install -q sherpa-onnx==1.12.38 soundfile numpy

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json, time
from pathlib import Path
import numpy as np
import soundfile as sf

DRIVE_DIR   = '/content/drive/MyDrive/asr_benchmark'
SAMPLES_DIR = Path(f'{DRIVE_DIR}/samples')
JSONL_PATH  = f'{DRIVE_DIR}/asr_benchmark_samples.jsonl'
MODEL_DIR   = f'{DRIVE_DIR}/model_data'
OUT_DIR     = Path(f'{DRIVE_DIR}/asr_rtf_results')
OUT_DIR.mkdir(exist_ok=True)

samples = []
with open(JSONL_PATH, encoding='utf-8') as f:
    for line in f:
        samples.append(json.loads(line))

print(f'Loaded {len(samples)} samples')
durations = [s['duration'] for s in samples]
print(f'Duration — min={min(durations):.1f}s  mean={np.mean(durations):.1f}s  max={max(durations):.1f}s')
print(f'Total   : {sum(durations)/60:.1f} min audio')

In [ ]:
# Check GPU
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                        capture_output=True, text=True)
print(result.stdout.strip())

In [ ]:
# Load model — inline Transcriber, chỉ cần model_data/ từ Drive
import numpy as np
import sherpa_onnx

PROVIDER    = 'cpu'   # đổi thành 'cuda' để test GPU
NUM_THREADS = 4
SAMPLE_RATE = 16_000

recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder = f'{MODEL_DIR}/encoder-epoch-20-avg-10.onnx',
    decoder = f'{MODEL_DIR}/decoder-epoch-20-avg-10.onnx',
    joiner  = f'{MODEL_DIR}/joiner-epoch-20-avg-10.onnx',
    tokens  = f'{MODEL_DIR}/config.json',
    num_threads     = NUM_THREADS,
    sample_rate     = SAMPLE_RATE,
    feature_dim     = 80,
    decoding_method = 'greedy_search',
    provider        = PROVIDER,
    debug           = False,
)

def transcribe(pcm_bytes: bytes, sr: int = SAMPLE_RATE) -> str:
    samples = np.frombuffer(pcm_bytes, dtype=np.int16).astype(np.float32) / 32768.0
    samples = np.concatenate([samples, np.zeros(sr // 2, dtype=np.float32)])
    stream = recognizer.create_stream()
    stream.accept_waveform(sr, samples)
    recognizer.decode_streams([stream])
    return stream.result.text.strip()

print(f'Model loaded (provider={PROVIDER}, threads={NUM_THREADS})')

In [ ]:
# Warmup + giới hạn 1000 samples để benchmark
N_WARMUP  = 10
N_BENCH   = 1000  # chỉ đo 1k samples, đủ để P95 ổn định

print(f'Warming up ({N_WARMUP} samples)...')
for s in samples[:N_WARMUP]:
    arr, sr = sf.read(str(SAMPLES_DIR / s['wav_file']), dtype='int16', always_2d=False)
    transcribe(arr.tobytes(), sr)
print('Done.')

bench_samples = samples[:N_BENCH]
print(f'Benchmark samples: {len(bench_samples)}')

In [ ]:
# Benchmark
MODEL_NAME = f'zipformer_rnnt_{PROVIDER}'

results = []
for i, s in enumerate(bench_samples):
    arr, sr = sf.read(str(SAMPLES_DIR / s['wav_file']), dtype='int16', always_2d=False)
    pcm = arr.tobytes()
    dur = s['duration']

    t0 = time.perf_counter()
    text = transcribe(pcm, sr)
    elapsed = time.perf_counter() - t0

    rtf = elapsed / dur if dur > 0 else 0.0
    results.append({
        'idx':        s['idx'],
        'wav_file':   s['wav_file'],
        'duration':   dur,
        'latency_ms': round(elapsed * 1000, 2),
        'rtf':        round(rtf, 4),
        'transcript': text,
        'ref':        s['sentence'],
    })

    if (i + 1) % 100 == 0:
        rtfs_so_far = [r['rtf'] for r in results]
        print(f'[{i+1}/{len(bench_samples)}]  RTF P50={np.median(rtfs_so_far):.4f}  P95={np.percentile(rtfs_so_far,95):.4f}')

print('\nBenchmark complete.')

In [ ]:
# Save per-sample JSONL
out_jsonl = OUT_DIR / f'{MODEL_NAME}.jsonl'
with open(out_jsonl, 'w', encoding='utf-8') as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')
print(f'Saved: {out_jsonl}')

In [ ]:
# Summary
rtfs = [r['rtf'] for r in results]
lats = [r['latency_ms'] for r in results]
durs = [r['duration'] for r in results]

summary = {
    'model':     MODEL_NAME,
    'n_samples': len(results),
    'rtf': {
        'mean':   round(float(np.mean(rtfs)),            4),
        'median': round(float(np.median(rtfs)),          4),
        'p95':    round(float(np.percentile(rtfs, 95)),  4),
        'p99':    round(float(np.percentile(rtfs, 99)),  4),
        'max':    round(float(np.max(rtfs)),             4),
    },
    'latency_ms': {
        'mean':   round(float(np.mean(lats)),           1),
        'median': round(float(np.median(lats)),         1),
        'p95':    round(float(np.percentile(lats, 95)), 1),
    },
    'audio_duration_s': {
        'mean':  round(float(np.mean(durs)),  2),
        'total': round(float(np.sum(durs)),   1),
    },
}

out_json = OUT_DIR / f'{MODEL_NAME}_summary.json'
with open(out_json, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print('=' * 50)
print(f"Model  : {summary['model']}")
print(f"Samples: {summary['n_samples']}")
print(f"RTF    — mean={summary['rtf']['mean']}  median={summary['rtf']['median']}  P95={summary['rtf']['p95']}  P99={summary['rtf']['p99']}")
print(f"Latency— mean={summary['latency_ms']['mean']}ms  P95={summary['latency_ms']['p95']}ms")
print('=' * 50)
print(f'Saved: {out_json}')

In [ ]:
# Plot RTF distribution
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# RTF histogram
axes[0].hist(rtfs, bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(np.median(rtfs), color='red',    linestyle='--', label=f'P50={np.median(rtfs):.3f}')
axes[0].axvline(np.percentile(rtfs, 95), color='orange', linestyle='--', label=f'P95={np.percentile(rtfs,95):.3f}')
axes[0].set_xlabel('RTF')
axes[0].set_ylabel('Count')
axes[0].set_title(f'RTF Distribution — {MODEL_NAME}')
axes[0].legend()

# RTF vs duration scatter
axes[1].scatter(durs, rtfs, alpha=0.3, s=5)
axes[1].set_xlabel('Audio duration (s)')
axes[1].set_ylabel('RTF')
axes[1].set_title('RTF vs Audio Duration')

plt.tight_layout()
out_png = OUT_DIR / f'{MODEL_NAME}_rtf_dist.png'
plt.savefig(str(out_png), dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_png}')